# 📦 Sandbox Compute with VideoDB

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/hackathon/guides/sandbox/sandbox_compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Use a VideoDB Sandbox as dedicated compute for scene indexing, voice generation, and image generation.

## 🧭 Flow

1. 🏗️ Create a sandbox and wait until it is ready.
2. 🎬 Run scene indexing on sandbox compute.
3. 🎙️ Generate OmniVoice audio, including voice cloning from YouTube.
4. 🖼️ Generate FLUX images.
5. 🛑 Stop the sandbox when done to end compute billing.

> If `sandbox_id` is omitted for self-inference models, VideoDB can auto-pick a compatible active sandbox.


## 🛠️ Setup

Install the SDK and connect to VideoDB.


In [ ]:
!pip install -q "git+https://github.com/Video-DB/videodb-python.git@hackathon"
!pip install -q python-dotenv


In [ ]:
import os
from getpass import getpass

from videodb import connect, SandboxModel, SandboxTier
from videodb._constants import SceneExtractionType

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect()
coll = conn.get_collection()
print("✅ Connected to VideoDB")


In [ ]:
video = coll.upload("https://www.youtube.com/watch?v=jeA-KBv0b68")

## 🏗️ 1. Create Sandbox

A sandbox is a warm compute pool for self-hosted inference models. Choose a tier based on the models you want to run.

| Tier | Good for |
|---|---|
| `small` | OmniVoice and smaller VLMs |
| `medium` | FLUX and larger VLMs |

## 🤖 Sandbox Models

VideoDB exposes sandbox-compatible models through the `SandboxModel` enum. Use these constants instead of raw model-name strings so the notebook stays aligned with the SDK.

| Use case | Model enum | Minimum tier | Notes |
|---|---|---|---|
| 🎬 Scene indexing | `SandboxModel.GEMMA_4_E2B` | `small` | Faster visual understanding model |
| 🎬 Scene indexing | `SandboxModel.QWEN_9B` | `small` | Smaller VLM option |
| 🎬 Scene indexing | `SandboxModel.GEMMA_4_26B` | `medium` | Higher quality visual understanding |
| 🎬 Scene indexing | `SandboxModel.QWEN_27B` | `medium` | Larger VLM option |
| 🎬 Scene indexing | `SandboxModel.GEMMA_4_31B` | `medium` | Best fit for this demo |
| 🎙️ Voice generation | `SandboxModel.OMNIVOICE` | `small` | Text-to-speech, voice design, and voice clone |
| 🖼️ Image generation | `SandboxModel.FLUX` | `medium` | FLUX image generation |

> This notebook creates a `medium` sandbox so all examples below can run on the same sandbox.


In [ ]:
# Create a sandbox (returns immediately in 'provisioning' state)
sandbox = conn.create_sandbox(tier=SandboxTier.medium)
print(f"Sandbox: {sandbox.id}, Status: {sandbox.status}, Tier: {sandbox.tier}")

In [ ]:
# Wait until the sandbox is active before running jobs.
sandbox.wait_for_ready(timeout=300, interval=5)
print(f"Sandbox ready: {sandbox.id}, Status: {sandbox.status}")

In [ ]:
sandbox.id

In [ ]:
# Manual polling alternative
sandbox.refresh()
print(f"Status: {sandbox.status}, Active: {sandbox.is_active}")

In [ ]:
# List all sandboxes
all_sandboxes = conn.list_sandboxes()
for sb in all_sandboxes:
    print(f"{sb.id} | {sb.name} | {sb.tier} | {sb.status}")

In [ ]:
# Get a specific sandbox by ID
sb = conn.get_sandbox(sandbox.id)
print(f"{sb.id} | {sb.status}")

## 🎬 2. Scene Indexing

Use `SandboxModel` enums and pass `sandbox_id=sandbox.id` to route visual indexing to the sandbox.


In [ ]:
video.id

### ▶️ Run scene indexing

Pass `sandbox_id=sandbox.id` to use your dedicated compute pool.


In [ ]:
index_id = video.index_scenes(
    extraction_type=SceneExtractionType.time_based,
    extraction_config={
        "time": 10,
        "select_frames": ["first"],
        "frame_count": 1,
    },
    model_name=SandboxModel.GEMMA_4_31B,
    prompt="Describe the scene in a clear, concise way.",
    sandbox_id=sandbox.id,
)
print(index_id)


In [ ]:
print(index_id)

In [ ]:
# Re-run until the index is ready.
idx = video.get_scene_index(index_id)
print(idx)
print(len(idx) if idx else 0)

In [ ]:
from videodb import play_stream

query = "fireship"
res = video.search(query)
shots = res.get_shots()

print(f"Found {len(shots)} result(s) for '{query}':")
for idx, shot in enumerate(shots, start=1):
    print(f"{idx}. {shot.start:.2f}s - {shot.end:.2f}s")

# Compile the search result into a playable highlight stream.
stream_url = res.compile()
player_url = f"https://player.videodb.io/watch?v={stream_url}"
print(f"Stream: {stream_url}")
print(f"Player: {player_url}")
play_stream(stream_url)

## 🎙️ 3. OmniVoice TTS

Use `coll.generate_voice(..., model_name=SandboxModel.OMNIVOICE, sandbox_id=sandbox.id)`.

The examples below show the main modes: basic TTS, voice design, voice clone, and extra config.


### 🔊 Helper: preview generated audio

This helper creates a simple timeline so you can listen to generated audio in the player.


In [ ]:
from videodb.editor import Timeline, Track, Clip, ImageAsset, AudioAsset, Fit

def demo_audio(audio_id):
    audio = coll.get_audio(audio_id)

    timeline = Timeline(conn)
    timeline.resolution = "1280x720"
    timeline.background = "#000000"

    audio_track = Track()
    audio_track.add_clip(0, Clip(asset=AudioAsset(id=audio_id), duration=float(audio.length)))
    timeline.add_track(audio_track)

    stream_url = timeline.generate_stream()
    player_url = f"https://player.videodb.io/watch?v={stream_url}"
    print(f"Stream: {stream_url}")
    print(f"Player: {player_url}")
    return player_url


### 🗣️ Basic TTS

In [ ]:
job = coll.generate_voice(
    text="Hello, welcome to VideoDB.",
    model_name=SandboxModel.OMNIVOICE,
    sandbox_id=sandbox.id,
)
print(job)

audio = job.wait(timeout=900, interval=5)
print(audio)
demo_audio(audio.id)


### 🎚️ Voice Design

In [ ]:
# Voice Design: pass instructions to control voice characteristics.
job = coll.generate_voice(
    text="Breaking news! Scientists discover a new planet.",
    model_name=SandboxModel.OMNIVOICE,
    sandbox_id=sandbox.id,
    config={
        "instructions": "A deep, authoritative male news anchor voice",
    },
)
print(job)

audio = job.wait(timeout=900, interval=5)
print(audio)
demo_audio(audio.id)


### 🧬 Voice Clone

In [ ]:
# Voice Clone: upload a reference audio, then use it for cloning.
ref_audio = coll.upload(
    url="https://www.youtube.com/shorts/7xOPzBhHKWY",
    media_type="audio",
)
print(f"Reference Audio ID: {ref_audio.id}, Length: {ref_audio.length}s")

job = coll.generate_voice(
    text="This is a cloned voice powered by OmniVoice. The future of self-hosted text to speech is here.",
    model_name=SandboxModel.OMNIVOICE,
    sandbox_id=sandbox.id,
    config={
        "ref_audio": ref_audio.generate_url(),
        "ref_text": "Sample reference text for the audio clip",
    },
)
print(job)

audio = job.wait(timeout=900, interval=5)
print(audio)
demo_audio(audio.id)


### ⚙️ Additional TTS Config

In [ ]:
# Additional config: speed, language, max_new_tokens.
job = coll.generate_voice(
    text="Hola, bienvenidos a VideoDB.",
    model_name=SandboxModel.OMNIVOICE,
    sandbox_id=sandbox.id,
    config={
        "speed": 1.2,
        "language": "es",
    },
)
print(job)

audio = job.wait(timeout=900, interval=5)
print(audio)


## 🖼️ 4. FLUX Image Generation

Use `coll.generate_image(..., model_name=SandboxModel.FLUX, sandbox_id=sandbox.id)`.


### 🌆 Basic FLUX Image

In [ ]:
job = coll.generate_image(
    prompt="A futuristic cityscape at sunset, neon lights reflecting off glass skyscrapers, cyberpunk style",
    model_name=SandboxModel.FLUX,
    sandbox_id=sandbox.id,
)
print(job)

image = job.wait(timeout=900, interval=5)
print(image)


In [ ]:
image = job.wait(timeout=900, interval=5)
print(image)

### 🎛️ FLUX with Config

In [ ]:
job = coll.generate_image(
    prompt="A photorealistic portrait of a robot reading a book in a cozy library",
    model_name=SandboxModel.FLUX,
    sandbox_id=sandbox.id,
    config={
        "size": "1024x1536",
        "num_inference_steps": 50,
        "guidance_scale": 4.0,
        "negative_prompt": "blurry, low quality, watermark",
    },
)
print(job)

image = job.wait(timeout=900, interval=5)
print(image)


## 🎞️ 5. Combine FLUX + OmniVoice

Generate an image and narration on the same sandbox, then compose them on a timeline.


In [ ]:
image_job = coll.generate_image(
    prompt="A dramatic mountain landscape at dawn, golden hour lighting, cinematic wide shot",
    model_name=SandboxModel.FLUX,
    sandbox_id=sandbox.id,
    config={"size": "1280x720", "num_inference_steps": 28},
)
image = image_job.wait(timeout=900, interval=5)
print(f"Image: {image.id}")

audio_job = coll.generate_voice(
    text="Witness the breathtaking beauty of dawn over the mountains. A new day begins.",
    model_name=SandboxModel.OMNIVOICE,
    sandbox_id=sandbox.id,
    config={
        "instructions": "female, young adult, moderate pitch, calm and cinematic",
    },
)
audio = audio_job.wait(timeout=900, interval=5)
print(f"Audio: {audio.id}, Length: {audio.length}s")


In [ ]:
timeline = Timeline(conn)
timeline.resolution = "1280x720"
timeline.background = "#000000"

image_track = Track()
image_track.add_clip(0, Clip(asset=ImageAsset(id=image.id), duration=float(audio.length), fit=Fit.crop))

audio_track = Track()
audio_track.add_clip(0, Clip(asset=AudioAsset(id=audio.id), duration=float(audio.length)))

timeline.add_track(image_track)
timeline.add_track(audio_track)

stream_url = timeline.generate_stream()
player_url = f"https://player.videodb.io/watch?v={stream_url}"
print(f"Stream: {stream_url}")
print(f"Player: {player_url}")


## 🛑 6. Stop Sandbox

Stop the sandbox when finished. Billing is based on sandbox runtime.


In [ ]:
sandbox.stop()
print(f"Sandbox {sandbox.id} status: {sandbox.status}")

In [ ]:
# Optionally wait for full teardown
sandbox.wait_for_stop(timeout=120)
print(f"Sandbox {sandbox.id} final status: {sandbox.status}")